In [1]:
import requests

url = "https://www.zimmo.be/fr/antwerpen-2000/a-vendre/appartement/LJ75K/?search=eyJmaWx0ZXIiOnsic3RhdHVzIjp7ImluIjpbIkZPUl9TQUxFIiwiVEFLRV9PVkVSIl19LCJwbGFjZUlkIjp7ImluIjpbMTEzXX0sImNhdGVnb3J5Ijp7ImluIjpbIkhPVVNFIiwiQVBBUlRNRU5UIl19fX0%3D&boosted=1#"

response = requests.get(url, headers={"User-Agent": "max-exercice"})
print(response)

<Response [200]>


In [2]:
from bs4 import BeautifulSoup


In [5]:
soup = BeautifulSoup(response.content, "html")


Objectifs:
unique_id(property id)
property_type
postal_code
city_name
address
livable_surface
total_land_surface
bedroom_count
build_year
peb_category((doing by calculating by their Primary Energy ConsumptionKwH/m/year)
garage
terrace
Province
lattitude
longitude
pre_school_distance
train_station_distance
supermarket_distance
price
swiming pool 

In [7]:
property_id = soup.find("div", attrs={"class": "zimmo-code"}).text.split(": ")[1][:-1]
print(property_id)

LJ75K


In [14]:
details = soup.find("section", attrs={"id": "main-features"})
print(details)

<section class="section-features align-block-left" id="main-features">
<div class="section-inner">
<div class="title-row">
<div class="section-title-block">
<h2 class="section-title __full-address">
<span>Minderbroedersstraat 6, 2000 Antwerpen</span>
<span class="adres link">
<!--?xml version="1.0" encoding="utf-8"?-->
<!-- Generator: Adobe Illustrator 24.2.1, SVG Export Plug-In . SVG Version: 6.00 Build 0)  -->
<svg class="svgicon svgicon-address-pinpoint-map" height="28" id="Layer_2" style="enable-background:new 0 0 56 56;" version="1.1" viewbox="0 0 56 56" width="28" x="0px" xml:space="preserve" xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" y="0px">
<g>
<polygon points="56,56 0,56 0,33.6 0,0 56,0 	" style="fill:#EAEDF2;"></polygon>
</g>
<g id="Layer_3">
<path d="M30.5,56l0.9-18c0,0,1.5-0.1,1.7-0.2c2.3-0.4,8-3,8-3H56v-5.7H43c-0.2-5.6-0.5-15.5,0.1-19.1
		c0.4-2.6,2.3-8.2,3.3-10.1l-7.7,0c-1.4,3.5-2.4,7.5-2.6,9.1c-0.6,4.4-0.3,15-0.1,20.1L31.1,31H0V38h25.7

In [21]:
address, rest = details.select("h2 span")[0].text.split(", ")
print(address)
postal_code, city_name = rest.split()
print(postal_code)
print(city_name)

Minderbroedersstraat 6
2000
Antwerpen


In [27]:
price = int(details.find("span", attrs={"class": "feature-value"}).text.strip().replace("€ ", "").replace(".", ""))
print(price)

447000


In [29]:
lis = details.find_all("li")
print(lis)

[<li class="show-on-print">
<strong class="feature-label show-on-print">Prix</strong>
<span class="feature-value">
                                € 447.000
                            </span>
</li>, <li class="show-on-print">
<strong class="feature-label">Adresse</strong>
<span class="feature-value">
<span>Minderbroedersstraat 6, 2000 Antwerpen</span>
</span>
</li>, <li>
<strong class="feature-label">Type</strong>
<span class="feature-value">App. rez-de-chaussée (Appartement)</span>
</li>, <li>
<strong class="feature-label">Surf. habitable</strong>
<span class="feature-value">
                                                                    92 m²
                                                            </span>
</li>, <li>
<strong class="feature-label">Chambres</strong>
<span class="feature-value">
                                                                    2
                                                            </span>
</li>, <li>
<strong class="feature-label">Sall

In [42]:
property_type = "House" if "Maison" in lis[2].text else "Appartment"
print(property_type)
total_land_surface = 0
livable_surface = int(lis[3].find("span").text.strip().split()[0])
if property_type == "House":
    print("h")
    total_land_surface = int(lis[4].find("span").text.strip().split()[0])
else:
    bedroom_count = int(lis[4].find("span").text.strip().split()[0])

print(livable_surface)
print(total_land_surface)
print(bedroom_count)


Appartment
92
0
2


In [56]:
for li in lis:
    cat = li.find("strong")
    if cat: cat = cat.text
    value = li.find("span")
    if value: value = value.text.strip()
    match cat:
        case "Prix":
            price = int(value.replace("€ ", "").replace(".", ""))
        case "Adresse":
            address, rest = value.split(", ")
            postal_code, city_name = rest.split()
        case "Type":
            property_type = "House" if "Maison" in value else "Appartment"
        case "Surf. habitable":
            livable_surface = int(value.split()[0])
        case "Sup. du terrain":
            total_land_surface = int(value.split()[0])
        case "Chambres":
            bedroom_count = int(value)
        case "Construit en":
            build_year = int(value)
        case "PEB":
            try:
                peb_category = int(value.split()[0])
            except ValueError:
                peb_category = None
            
if not total_land_surface:
    total_land_surface = livable_surface
print(price)
print(address)
print(postal_code, city_name)
print(property_type)
print(livable_surface)
print(bedroom_count)
print(build_year)
print(peb_category)


447000
Minderbroedersstraat 6
2000 Antwerpen
Appartment
92
2
2009
194


Missing: Terrace, Garage, Lat, Long, Swimming pool

In [60]:
coordinates = soup.find("zmaps-estate-detail")
latitude = coordinates.get("lat")
longitude = coordinates.get("long")
print(latitude, longitude)

51.2231264 4.4056312
